In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import time
import math
import random
import torch
from torch import nn
from torch.utils import data
import torch.utils.data as Data
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_curve
from sklearn.metrics import roc_auc_score, auc, precision_score, recall_score, matthews_corrcoef, roc_curve, auc

def setup_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
setup_seed(42)

C:\Users\dell\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\dell\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
class Embeddings(nn.Module):
    def __init__(self, vocab_size, num_hiddens, dropout, max_len):
        super(Embeddings, self).__init__()
        self.token_embedding = nn.Embedding(vocab_size, num_hiddens)
        self.segment_embedding = nn.Embedding(3, num_hiddens)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, num_hiddens))
        
    def forward(self, tokens, segments):
        X = self.token_embedding(tokens) + self.segment_embedding(segments) + self.pos_embedding.data[:, :tokens.shape[1], :]
        return X

In [3]:
def sequence_mask(X, valid_len, value=0):
    maxlen = X.size(1)
    mask = torch.arange((maxlen), dtype=torch.float32, device=X.device)[None, :].to(device) < valid_len[:, None].to(device)
    X[~mask] = value
    return X

In [4]:
def masked_softmax(X, valid_lens):
    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            valid_lens = torch.repeat_interleave(valid_lens, shape[1])
        else:
            valid_lens = valid_lens.reshape(-1)
        X = sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6)
        return nn.functional.softmax(X.reshape(shape), dim=-1)

In [5]:
def transpose_qkv(X, num_heads):
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
    X = X.permute(0, 2, 1, 3)
    return X.reshape(-1, X.shape[2], X.shape[3])

In [6]:
def transpose_output(X, num_heads):
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)

In [7]:
class DotProductAttention(nn.Module):
    def __init__(self, dropout, **kwargs):
        super(DotProductAttention, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)

In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, key_size, query_size, value_size,
                 num_hiddens, num_heads, dropout, bias=False, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.attention = DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)
        
    def forward(self, queries, keys, values, valid_lens):
        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys = transpose_qkv(self.W_k(keys), self.num_heads)
        values = transpose_qkv(self.W_v(values), self.num_heads)
        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(valid_lens, repeats=self.num_heads, dim=0)
        output = self.attention(queries, keys, values, valid_lens)
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat)

In [9]:
class PositionWiseFFN(nn.Module):
    def __init__(self, ffn_num_input, ffn_num_hiddens, ffn_num_outputs, **kwargs):
        super(PositionWiseFFN, self).__init__(**kwargs)
        self.dense1 = nn.Linear(ffn_num_input, ffn_num_hiddens)
        self.relu = nn.LeakyReLU()
        self.dense2 = nn.Linear(ffn_num_hiddens, ffn_num_outputs)
        
    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))

In [10]:
class AddNorm(nn.Module):
    def __init__(self, normalized_shape, dropout, **kwargs):
        super(AddNorm, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(normalized_shape)
        
    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)

In [11]:
class EncoderBlock(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens, 
                 norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
                 dropout_addnorm, dropout_attention, use_bias=False, **kwargs):
        super(EncoderBlock, self).__init__(**kwargs)
        self.attention = MultiHeadAttention(key_size, query_size, value_size, 
                                            num_hiddens, num_heads, 
                                            dropout_attention, use_bias)
        self.addnorm1 = AddNorm(norm_shape, dropout_addnorm)
        self.ffn = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(norm_shape, dropout_addnorm)
        
    def forward(self, X, valid_lens):
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        return self.addnorm2(Y, self.ffn(Y))

In [12]:
class Encoder(nn.Module):
    def __init__(self, 
                 key_size, query_size, value_size, vocab_size, num_hiddens,
                 norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_layers, 
                 dropout_embedding, dropout_addnorm, dropout_attention, 
                 use_bias=False, **kwargs):
        super(Encoder, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.embedding = Embeddings(vocab_size, num_hiddens, dropout_embedding, max_len=200)
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module(
                "block" + str(i),
                EncoderBlock(key_size, query_size, value_size, num_hiddens,
                             norm_shape, ffn_num_input, ffn_num_hiddens, num_heads, 
                             dropout_addnorm, dropout_attention, use_bias))
            
    def forward(self, tokens, segments, valid_lens, *args):
        X = self.embedding(tokens, segments) * math.sqrt(self.num_hiddens)
        X = X.to(torch.float32)
        self.attention_weights = [None] * len(self.blks)
        for i, blk in enumerate(self.blks):
            X = blk(X, valid_lens)
            self.attention_weights[i] = blk.attention.attention.attention_weights
        return X

In [13]:
class DecoderBlock(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens, 
                 norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
                 dropout_addnorm, dropout_attention, use_bias=False, **kwargs):
        super(DecoderBlock, self).__init__(**kwargs)
        self.attention = MultiHeadAttention(key_size, query_size, value_size, 
                                            num_hiddens, num_heads, 
                                            dropout_attention, use_bias)
        self.addnorm1 = AddNorm(norm_shape, dropout_addnorm)
        self.ffn = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(norm_shape, dropout_addnorm)
        
    def forward(self, X, valid_lens):
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        return self.addnorm2(Y, self.ffn(Y))

In [14]:
class Decoder(nn.Module):
    def __init__(self, 
                 key_size, query_size, value_size, num_hiddens,
                 norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_layers, 
                 dropout_embedding, dropout_addnorm, dropout_attention, 
                 use_bias=False, max_len=200, **kwargs):
        super(Decoder, self).__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, num_hiddens))
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module(
                "block" + str(i),
                DecoderBlock(key_size, query_size, value_size, num_hiddens,
                             norm_shape, ffn_num_input, ffn_num_hiddens, num_heads, 
                             dropout_addnorm, dropout_attention, use_bias))
            
    def forward(self, X, valid_lens):
        X = X + self.pos_embedding.data[:, :X.shape[1], :]
        X = X.to(torch.float32)
        self.attention_weights = [None] * len(self.blks)
        for i, blk in enumerate(self.blks):
            X = blk(X, valid_lens)
            self.attention_weights[i] = blk.attention.attention.attention_weights
        return X

In [15]:
class pMHC_net(nn.Module):
    def __init__(self, **kwargs):
        super(pMHC_net, self).__init__()
        # Encoder
        self.encoder_MHC = Encoder(key_size=64, query_size=64, value_size=64, vocab_size=64, num_hiddens=64,
                                   norm_shape=[64], ffn_num_input=64, ffn_num_hiddens=128,
                                   num_heads=2, num_layers=1, 
                                   dropout_embedding=0.01, dropout_addnorm=0.01, dropout_attention=0.01, 
                                   use_bias=False)
        self.encoder_peptide = Encoder(key_size=64, query_size=64, value_size=64, vocab_size=64, num_hiddens=64,
                                       norm_shape=[64], ffn_num_input=64, ffn_num_hiddens=128,
                                       num_heads=2, num_layers=1, 
                                       dropout_embedding=0.01, dropout_addnorm=0.01, dropout_attention=0.01, 
                                       use_bias=False)
        self.pMHC_decoder = Decoder(key_size=64, query_size=64, value_size=64, vocab_size=64, num_hiddens=64,
                                    norm_shape=[64], ffn_num_input=64, ffn_num_hiddens=128,
                                    num_heads=1, num_layers=1, 
                                    dropout_embedding=0.01, dropout_addnorm=0.01, dropout_attention=0.01, 
                                    use_bias=False)
        self.fnc = nn.Sequential(nn.Linear(3200, 1028),
                                 nn.ReLU(),
                                 nn.BatchNorm1d(1028),
                                 nn.Linear(1028, 2))
        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, 
                tokens_MHC_res, segments_MHC_res, valid_len_MHC_res,
                tokens_peptide_res, segments_peptide_res, valid_len_peptide_res,
                valid_len):
        MHC = self.encoder_MHC(tokens_MHC_res, segments_MHC_res, valid_len_MHC_res)
        peptide = self.encoder_peptide(tokens_peptide_res, segments_peptide_res, valid_len_peptide_res)
        X_encoder = torch.cat((peptide, MHC), dim=1)
        X_decoder = self.pMHC_decoder(X_encoder, valid_len)
        X = self.fnc(X_decoder.reshape(X_decoder.shape[0], -1))
        return self.softmax(X), X_encoder, X_decoder

In [16]:
class MaskLM(nn.Module):
    """The masked language model task of BERT."""
    def __init__(self, vocab_size, num_hiddens, num_inputs, **kwargs):
        super(MaskLM, self).__init__(**kwargs)
        self.mlp = nn.Sequential(nn.Linear(num_inputs, num_hiddens),
                                 nn.ReLU(), 
                                 nn.LayerNorm(num_hiddens),
                                 nn.Linear(num_hiddens, vocab_size))
        
    def forward(self, X, pred_positions):
        num_pred_positions = pred_positions.shape[1]
        pred_positions = pred_positions.reshape(-1)
        batch_size = X.shape[0]
        batch_idx = torch.arange(0, batch_size)
        # Suppose that `batch_size` = 2, `num_pred_positions` = 3, then
        # `batch_idx` is `torch.tensor([0, 0, 0, 1, 1, 1])`
        batch_idx = torch.repeat_interleave(batch_idx, num_pred_positions)
        masked_X = X[batch_idx, pred_positions]
        masked_X = masked_X.reshape((batch_size, num_pred_positions, -1))
        mlm_Y_hat = self.mlp(masked_X)
        return mlm_Y_hat

In [17]:
class NextSentencePred(nn.Module):
    def __init__(self, num_inputs, **kwargs):
        super(NextSentencePred, self).__init__(**kwargs)
        self.output = nn.Linear(num_inputs, 2)

    def forward(self, X):
        return self.output(X)

In [18]:
class Transformer(nn.Module):
    def __init__(self, **kwargs):
        super(Transformer, self).__init__()
        # Encoder
        self.encoder = Encoder(key_size=64, query_size=64, value_size=64, vocab_size=24, num_hiddens=64,
                               norm_shape=[64], ffn_num_input=64, ffn_num_hiddens=128,
                               num_heads=2, num_layers=2, 
                               dropout_embedding=0.01, dropout_addnorm=0.01, dropout_attention=0.01, 
                               use_bias=False)
        self.hidden = nn.Sequential(nn.Linear(64, 64),
                                    nn.Tanh())
        self.mlm = MaskLM(vocab_size=24, num_hiddens=64, num_inputs=64)
        self.nsp = NextSentencePred(num_inputs=64)
        
    def forward(self, tokens, segments, valid_len, pred_positions=None):
        encoded_X = self.encoder(tokens, segments, valid_len)
        if pred_positions is not None:
            mlm_Y_hat = self.mlm(encoded_X, pred_positions)
        else:
            mlm_Y_hat = None
        nsp_Y_hat = self.nsp(self.hidden(encoded_X[:, 0, :]))
        return encoded_X, mlm_Y_hat, nsp_Y_hat

In [19]:
class TCR_net(nn.Module):
    def __init__(self, **kwargs):
        super(TCR_net, self).__init__()
        # Decoder
        self.decoder = Decoder(key_size=64, query_size=64, value_size=64, vocab_size=64, num_hiddens=64,
                               norm_shape=[64], ffn_num_input=64, ffn_num_hiddens=128,
                               num_heads=1, num_layers=2, 
                               dropout_embedding=0, dropout_addnorm=0, dropout_attention=0, 
                               use_bias=False)
        # Classify Head
        self.fnc = nn.Sequential(nn.Linear(6208, 2048),
                                 nn.ReLU(),
                                 nn.BatchNorm1d(2048),
                                 #nn.Dropout(p=0.1),
                                 nn.Linear(2048, 64),
                                 nn.ReLU(),
                                 nn.BatchNorm1d(64),
                                 nn.Linear(64, 2))
        self.softmax = nn.Softmax(dim=1)

    def forward(self, encoder_TCR, encoder_pMHC, valid_len_pMT_res):
        # TCR-pMHC
        X = torch.cat((encoder_TCR, encoder_pMHC), dim=1)
        # Decoder
        X = self.decoder(X, valid_len_pMT_res)
        # Classify Head
        print(X.reshape(X.shape[0], -1).shape)
        X = self.fnc(X.reshape(X.shape[0], -1))
        return self.softmax(X)

In [20]:
def get_tokens_and_segments(tokens_a, tokens_b=None):
    tokens = ['<cls>'] + tokens_a + ['<sep>']
    # 0 and 1 are marking segment A and B, respectively
    segments = [0] * (len(tokens_a) + 2)
    if tokens_b is not None:
        tokens += tokens_b + ['<sep>']
        segments += [1] * (len(tokens_b) + 1)
    return tokens, segments

In [21]:
class TCR_DataSet(Data.Dataset):
    def __init__(self, data):
        super(TCR_DataSet, self).__init__()
        vocab = {"A": 0, "R": 1, "N": 2, "D": 3, "C": 4, "Q": 5, "E": 6, "G": 7, "H": 8, "I": 9,
                 "L": 10, "K": 11, "M": 12, "F": 13, "P": 14, "S": 15, "T": 16, "W": 17, "Y": 18, "V": 19,
                 "<cls>": 20, "<sep>": 21, "<pad>": 22}
        TRB = data["TCRB"].to_list()
        TRA = data["TCRA"].to_list()
        MHC = data["MHC"].to_list()
        peptide = data["Peptide"].to_list()
        labels = data["labels"].to_list()
        tokens_TCR_res = []
        segments_TCR_res = []
        valid_len_TCR_res = []
        tokens_MHC_res = []
        segments_MHC_res = []
        valid_len_MHC_res = []
        tokens_peptide_res = []
        segments_peptide_res = []
        valid_len_peptide_res = []
        valid_len_pMHC_res = []
        valid_len_pMT_res = []
        for i in range(len(TRB)):
            # TCR
            if TRA[i] == "None":
                tokens_TCR, segments_TCR = get_tokens_and_segments(list(TRB[i]), None)
            else:
                tokens_TCR, segments_TCR = get_tokens_and_segments(list(TRB[i]), list(TRA[i]))
            valid_len_TCR_res.append(len(tokens_TCR))
            tokens_TCR = tokens_TCR + ["<pad>"] * (47 - len(tokens_TCR))
            tokens_TCR = [vocab[a] for a in tokens_TCR]
            segments_TCR = segments_TCR + [2] * (47 - len(segments_TCR))
            tokens_TCR_res.append(tokens_TCR)
            segments_TCR_res.append(segments_TCR)
            # MHC
            tokens_MHC, segments_MHC = get_tokens_and_segments(list(MHC[i]), None)
            valid_len_MHC_res.append(len(tokens_MHC))
            tokens_MHC = tokens_MHC + ["<pad>"] * (36 - len(tokens_MHC))
            tokens_MHC = [vocab[a] for a in tokens_MHC]
            tokens_MHC_res.append(tokens_MHC)
            segments_MHC = segments_MHC + [2] * (36 - len(segments_MHC))
            segments_MHC_res.append(segments_MHC)
            # peptide
            tokens_peptide, segments_peptide = get_tokens_and_segments(list(peptide[i]), None)
            valid_len_peptide_res.append(len(tokens_peptide))
            tokens_peptide = tokens_peptide + ["<pad>"] * (14 - len(tokens_peptide))
            tokens_peptide = [vocab[a] for a in tokens_peptide]
            tokens_peptide_res.append(tokens_peptide)
            segments_peptide = segments_peptide + [2] * (14 - len(segments_peptide))
            segments_peptide_res.append(segments_peptide)
            # decoder
            valid_len_pMT_res.append(97)
            valid_len_pMHC_res.append(50)
        self.tokens_TCR_res = torch.tensor([tokens_TCR_res]).reshape(-1, 47)
        self.segments_TCR_res = torch.tensor([segments_TCR_res]).reshape(-1, 47)
        self.valid_len_TCR_res = torch.tensor(valid_len_TCR_res)
        self.tokens_MHC_res = torch.tensor([tokens_MHC_res]).reshape(-1, 36)
        self.segments_MHC_res = torch.tensor([segments_MHC_res]).reshape(-1, 36)
        self.valid_len_MHC_res = torch.tensor(valid_len_MHC_res)
        self.tokens_peptide_res = torch.tensor([tokens_peptide_res]).reshape(-1, 14)
        self.segments_peptide_res = torch.tensor([segments_peptide_res]).reshape(-1, 14)
        self.valid_len_peptide_res = torch.tensor(valid_len_peptide_res)
        self.valid_len_pMT_res = torch.tensor(valid_len_pMT_res)
        self.valid_len_pMHC_res = torch.tensor(valid_len_pMHC_res)
        self.labels = torch.tensor(labels)

    def __len__(self):
        return self.tokens_TCR_res.shape[0]

    def __getitem__(self, idx):
        return [self.tokens_TCR_res[idx], self.segments_TCR_res[idx], self.valid_len_TCR_res[idx], 
                 self.tokens_MHC_res[idx], self.segments_MHC_res[idx], self.valid_len_MHC_res[idx],
                 self.tokens_peptide_res[idx], self.segments_peptide_res[idx], self.valid_len_peptide_res[idx],
                 self.valid_len_pMT_res[idx], self.valid_len_pMHC_res[idx]], self.labels[idx]

In [22]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

TCR_encoder = Transformer()
TCR_encoder.load_state_dict(torch.load("../model/TCR_pretrain_model.pkl", map_location=torch.device('cpu')))
TCR_encoder = TCR_encoder.to(device)
TCR_encoder.eval()

pMHC_encoder = pMHC_net()
pMHC_encoder.load_state_dict(torch.load("../model/pMHC_pretrain_model.pkl", map_location=torch.device('cpu')))
pMHC_encoder = pMHC_encoder.to(device)
pMHC_encoder.eval()

model = TCR_net()
model.load_state_dict(torch.load("../model/TransTCR_model.pkl", map_location=torch.device('cpu')))
model = model.to(device)
model.eval()

print("Model loading completed")

cpu
Model loading completed


In [23]:
test = pd.read_csv("./TransTCR_test/TransTCR_beta_test.csv").fillna("None")
test_data_iter = Data.DataLoader(TCR_DataSet(test), batch_size = len(test["TCRB"].to_list()), shuffle = False)
for step, (data, label) in enumerate(test_data_iter):
    tokens_TCR_res = data[0]
    segments_TCR_res = data[1]
    valid_len_TCR_res = data[2]
    tokens_MHC_res = data[3]
    segments_MHC_res = data[4]
    valid_len_MHC_res = data[5]
    tokens_peptide_res = data[6]
    segments_peptide_res = data[7]
    valid_len_peptide_res = data[8]
    valid_len_pMT_res = data[9]
    valid_len_pMHC_res = data[10]
    labels = label
    model.to(device)
    model.eval()
    with torch.no_grad():
        encoder_TCR, _, _ = TCR_encoder(tokens_TCR_res.to(device), 
                                        segments_TCR_res.to(device), 
                                        valid_len_TCR_res.to(device), 
                                        None)
        _, encoder_pMHC, decoder_pMHC = pMHC_encoder(tokens_MHC_res.to(device), 
                                                     segments_MHC_res.to(device), 
                                                     valid_len_MHC_res.to(device),
                                                     tokens_peptide_res.to(device), 
                                                     segments_peptide_res.to(device), 
                                                     valid_len_peptide_res.to(device),
                                                     valid_len_pMHC_res.to(device))
        output = model(encoder_TCR.to(device), encoder_pMHC.to(device), valid_len_pMT_res.to(device))
        output_softmax = output
        labels = labels.to(device)
        _, output = torch.max(output, 1)
conf_mat = confusion_matrix(labels, output)
confusion_matrix1 = confusion_matrix(labels, output)
new_confusion_matrix1 = [[confusion_matrix1[1,1], confusion_matrix1[1,0]],
                         [confusion_matrix1[0,1], confusion_matrix1[0,0]]]
accuracy = accuracy_score(labels, output) #准确率
precision = precision_score(labels, output) #精确率
recall = recall_score(labels, output) #召回率
f1= f1_score(labels, output) #F1
MCC = matthews_corrcoef(labels, output) #MCC
fpr, tpr, threshold = roc_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_auc = auc(fpr, tpr)
precision1, recall1, threshold = precision_recall_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_aupr = auc(recall1, precision1)
print("-----------------------------------------------------------------")
print("beta")
print(new_confusion_matrix1)
print("accuracy, precision, recall, f1, MCC, AUC, AUPR")
print(accuracy, precision, recall, f1, MCC, roc_auc, roc_aupr)

test = pd.read_csv("./TransTCR_test/TransTCR_beta_test.csv")
test["Prediction"] = output
test["Score"] = output_softmax[:,1].to("cpu").detach().numpy()
pd.DataFrame(test).to_csv("./TransTCR_test/TransTCR_beta_test.csv", index=False)

torch.Size([850, 6208])
-----------------------------------------------------------------
beta
[[343, 82], [61, 364]]
accuracy, precision, recall, f1, MCC, AUC, AUPR
0.831764705882353 0.849009900990099 0.8070588235294117 0.827503015681544 0.6643409090252683 0.8059211072664361 0.8271887735750929


In [24]:
test = pd.read_csv("./TransTCR_test/TransTCR_pair_test.csv").fillna("None")
test_data_iter = Data.DataLoader(TCR_DataSet(test), batch_size = len(test["TCRB"].to_list()), shuffle = False)
for step, (data, label) in enumerate(test_data_iter):
    tokens_TCR_res = data[0]
    segments_TCR_res = data[1]
    valid_len_TCR_res = data[2]
    tokens_MHC_res = data[3]
    segments_MHC_res = data[4]
    valid_len_MHC_res = data[5]
    tokens_peptide_res = data[6]
    segments_peptide_res = data[7]
    valid_len_peptide_res = data[8]
    valid_len_pMT_res = data[9]
    valid_len_pMHC_res = data[10]
    labels = label
    model.to(device)
    model.eval()
    with torch.no_grad():
        encoder_TCR, _, _ = TCR_encoder(tokens_TCR_res.to(device), 
                                        segments_TCR_res.to(device), 
                                        valid_len_TCR_res.to(device), 
                                        None)
        _, encoder_pMHC, decoder_pMHC = pMHC_encoder(tokens_MHC_res.to(device), 
                                                     segments_MHC_res.to(device), 
                                                     valid_len_MHC_res.to(device),
                                                     tokens_peptide_res.to(device), 
                                                     segments_peptide_res.to(device), 
                                                     valid_len_peptide_res.to(device),
                                                     valid_len_pMHC_res.to(device))
        output = model(encoder_TCR.to(device), encoder_pMHC.to(device), valid_len_pMT_res.to(device))
        output_softmax = output
        labels = labels.to(device)
        _, output = torch.max(output, 1)
conf_mat = confusion_matrix(labels, output)
confusion_matrix1 = confusion_matrix(labels, output)
new_confusion_matrix1 = [[confusion_matrix1[1,1], confusion_matrix1[1,0]],
                         [confusion_matrix1[0,1], confusion_matrix1[0,0]]]
accuracy = accuracy_score(labels, output) #准确率
precision = precision_score(labels, output) #精确率
recall = recall_score(labels, output) #召回率
f1= f1_score(labels, output) #F1
MCC = matthews_corrcoef(labels, output) #MCC
fpr, tpr, threshold = roc_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_auc = auc(fpr, tpr)
precision1, recall1, threshold = precision_recall_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_aupr = auc(recall1, precision1)
print("-----------------------------------------------------------------")
print("beta common")
print(new_confusion_matrix1)
print("accuracy, precision, recall, f1, MCC, AUC, AUPR")
print(accuracy, precision, recall, f1, MCC, roc_auc, roc_aupr)

test = pd.read_csv("./TransTCR_test/TransTCR_pair_test.csv")
test["Prediction"] = output
test["Score"] = output_softmax[:,1].to("cpu").detach().numpy()
pd.DataFrame(test).to_csv("./TransTCR_test/TransTCR_pair_test.csv", index=False)

torch.Size([212, 6208])
-----------------------------------------------------------------
beta common
[[95, 11], [21, 85]]
accuracy, precision, recall, f1, MCC, AUC, AUPR
0.8490566037735849 0.8189655172413793 0.8962264150943396 0.8558558558558559 0.7012406903104856 0.9085973656105377 0.9283564448253434


In [25]:
test = pd.read_csv("./TransTCR_test/McPAS/beta_major.csv").fillna("None")
test_data_iter = Data.DataLoader(TCR_DataSet(test), batch_size = len(test["TCRB"].to_list()), shuffle = False)
for step, (data, label) in enumerate(test_data_iter):
    tokens_TCR_res = data[0]
    segments_TCR_res = data[1]
    valid_len_TCR_res = data[2]
    tokens_MHC_res = data[3]
    segments_MHC_res = data[4]
    valid_len_MHC_res = data[5]
    tokens_peptide_res = data[6]
    segments_peptide_res = data[7]
    valid_len_peptide_res = data[8]
    valid_len_pMT_res = data[9]
    valid_len_pMHC_res = data[10]
    labels = label
    model.to(device)
    model.eval()
    with torch.no_grad():
        encoder_TCR, _, _ = TCR_encoder(tokens_TCR_res.to(device), 
                                        segments_TCR_res.to(device), 
                                        valid_len_TCR_res.to(device), 
                                        None)
        _, encoder_pMHC, decoder_pMHC = pMHC_encoder(tokens_MHC_res.to(device), 
                                                     segments_MHC_res.to(device), 
                                                     valid_len_MHC_res.to(device),
                                                     tokens_peptide_res.to(device), 
                                                     segments_peptide_res.to(device), 
                                                     valid_len_peptide_res.to(device),
                                                     valid_len_pMHC_res.to(device))
        output = model(encoder_TCR.to(device), encoder_pMHC.to(device), valid_len_pMT_res.to(device))
        output_softmax = output
        labels = labels.to(device)
        _, output = torch.max(output, 1)
conf_mat = confusion_matrix(labels, output)
confusion_matrix1 = confusion_matrix(labels, output)
new_confusion_matrix1 = [[confusion_matrix1[1,1], confusion_matrix1[1,0]],
                         [confusion_matrix1[0,1], confusion_matrix1[0,0]]]
accuracy = accuracy_score(labels, output) #准确率
precision = precision_score(labels, output) #精确率
recall = recall_score(labels, output) #召回率
f1= f1_score(labels, output) #F1
MCC = matthews_corrcoef(labels, output) #MCC
fpr, tpr, threshold = roc_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_auc = auc(fpr, tpr)
precision1, recall1, threshold = precision_recall_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_aupr = auc(recall1, precision1)
print("-----------------------------------------------------------------")
print("beta uncommon")
print(new_confusion_matrix1)
print("accuracy, precision, recall, f1, MCC, AUC, AUPR")
print(accuracy, precision, recall, f1, MCC, roc_auc, roc_aupr)

test = pd.read_csv("./TransTCR_test/McPAS/beta_major.csv")
test["Prediction"] = output
test["Score"] = output_softmax[:,1].to("cpu").detach().numpy()
pd.DataFrame(test).to_csv("./TransTCR_test/McPAS/beta_major.csv", index=False)

torch.Size([2402, 6208])
-----------------------------------------------------------------
beta uncommon
[[753, 448], [209, 992]]
accuracy, precision, recall, f1, MCC, AUC, AUPR
0.7264779350541216 0.7827442827442828 0.626977518734388 0.6962552011095701 0.4622001878288386 0.7627102310661182 0.7710410324442136


In [26]:
test = pd.read_csv("./TransTCR_test/McPAS/beta_few.csv").fillna("None")
test_data_iter = Data.DataLoader(TCR_DataSet(test), batch_size = len(test["TCRB"].to_list()), shuffle = False)
for step, (data, label) in enumerate(test_data_iter):
    tokens_TCR_res = data[0]
    segments_TCR_res = data[1]
    valid_len_TCR_res = data[2]
    tokens_MHC_res = data[3]
    segments_MHC_res = data[4]
    valid_len_MHC_res = data[5]
    tokens_peptide_res = data[6]
    segments_peptide_res = data[7]
    valid_len_peptide_res = data[8]
    valid_len_pMT_res = data[9]
    valid_len_pMHC_res = data[10]
    labels = label
    model.to(device)
    model.eval()
    with torch.no_grad():
        encoder_TCR, _, _ = TCR_encoder(tokens_TCR_res.to(device), 
                                        segments_TCR_res.to(device), 
                                        valid_len_TCR_res.to(device), 
                                        None)
        _, encoder_pMHC, decoder_pMHC = pMHC_encoder(tokens_MHC_res.to(device), 
                                                     segments_MHC_res.to(device), 
                                                     valid_len_MHC_res.to(device),
                                                     tokens_peptide_res.to(device), 
                                                     segments_peptide_res.to(device), 
                                                     valid_len_peptide_res.to(device),
                                                     valid_len_pMHC_res.to(device))
        output = model(encoder_TCR.to(device), encoder_pMHC.to(device), valid_len_pMT_res.to(device))
        output_softmax = output
        labels = labels.to(device)
        _, output = torch.max(output, 1)
conf_mat = confusion_matrix(labels, output)
confusion_matrix1 = confusion_matrix(labels, output)
new_confusion_matrix1 = [[confusion_matrix1[1,1], confusion_matrix1[1,0]],
                         [confusion_matrix1[0,1], confusion_matrix1[0,0]]]
accuracy = accuracy_score(labels, output) #准确率
precision = precision_score(labels, output) #精确率
recall = recall_score(labels, output) #召回率
f1= f1_score(labels, output) #F1
MCC = matthews_corrcoef(labels, output) #MCC
fpr, tpr, threshold = roc_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_auc = auc(fpr, tpr)
precision1, recall1, threshold = precision_recall_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_aupr = auc(recall1, precision1)
print("-----------------------------------------------------------------")
print("pair")
print(new_confusion_matrix1)
print("accuracy, precision, recall, f1, MCC, AUC, AUPR")
print(accuracy, precision, recall, f1, MCC, roc_auc, roc_aupr)

test = pd.read_csv("./TransTCR_test/McPAS/beta_few.csv")
test["Prediction"] = output
test["Score"] = output_softmax[:,1].to("cpu").detach().numpy()
pd.DataFrame(test).to_csv("./TransTCR_test/McPAS/beta_few.csv", index=False)

torch.Size([2052, 6208])
-----------------------------------------------------------------
pair
[[996, 30], [533, 493]]
accuracy, precision, recall, f1, MCC, AUC, AUPR
0.7256335282651072 0.6514061478090255 0.9707602339181286 0.7796477495107632 0.5177575062881281 0.8226177855294508 0.7438436288784362


In [27]:
test = pd.read_csv("./TransTCR_test/McPAS/beta_zero.csv").fillna("None")
test_data_iter = Data.DataLoader(TCR_DataSet(test), batch_size = len(test["TCRB"].to_list()), shuffle = False)
for step, (data, label) in enumerate(test_data_iter):
    tokens_TCR_res = data[0]
    segments_TCR_res = data[1]
    valid_len_TCR_res = data[2]
    tokens_MHC_res = data[3]
    segments_MHC_res = data[4]
    valid_len_MHC_res = data[5]
    tokens_peptide_res = data[6]
    segments_peptide_res = data[7]
    valid_len_peptide_res = data[8]
    valid_len_pMT_res = data[9]
    valid_len_pMHC_res = data[10]
    labels = label
    model.to(device)
    model.eval()
    with torch.no_grad():
        encoder_TCR, _, _ = TCR_encoder(tokens_TCR_res.to(device), 
                                        segments_TCR_res.to(device), 
                                        valid_len_TCR_res.to(device), 
                                        None)
        _, encoder_pMHC, decoder_pMHC = pMHC_encoder(tokens_MHC_res.to(device), 
                                                     segments_MHC_res.to(device), 
                                                     valid_len_MHC_res.to(device),
                                                     tokens_peptide_res.to(device), 
                                                     segments_peptide_res.to(device), 
                                                     valid_len_peptide_res.to(device),
                                                     valid_len_pMHC_res.to(device))
        output = model(encoder_TCR.to(device), encoder_pMHC.to(device), valid_len_pMT_res.to(device))
        output_softmax = output
        labels = labels.to(device)
        _, output = torch.max(output, 1)
conf_mat = confusion_matrix(labels, output)
confusion_matrix1 = confusion_matrix(labels, output)
new_confusion_matrix1 = [[confusion_matrix1[1,1], confusion_matrix1[1,0]],
                         [confusion_matrix1[0,1], confusion_matrix1[0,0]]]
accuracy = accuracy_score(labels, output) #准确率
precision = precision_score(labels, output) #精确率
recall = recall_score(labels, output) #召回率
f1= f1_score(labels, output) #F1
MCC = matthews_corrcoef(labels, output) #MCC
fpr, tpr, threshold = roc_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_auc = auc(fpr, tpr)
precision1, recall1, threshold = precision_recall_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_aupr = auc(recall1, precision1)
print("-----------------------------------------------------------------")
print("pair common")
print(new_confusion_matrix1)
print("accuracy, precision, recall, f1, MCC, AUC, AUPR")
print(accuracy, precision, recall, f1, MCC, roc_auc, roc_aupr)

test = pd.read_csv("./TransTCR_test/McPAS/beta_zero.csv")
test["Prediction"] = output
test["Score"] = output_softmax[:,1].to("cpu").detach().numpy()
pd.DataFrame(test).to_csv("./TransTCR_test/McPAS/beta_zero.csv", index=False)

torch.Size([338, 6208])
-----------------------------------------------------------------
pair common
[[110, 59], [34, 135]]
accuracy, precision, recall, f1, MCC, AUC, AUPR
0.7248520710059172 0.7638888888888888 0.650887573964497 0.7028753993610224 0.4547068337912341 0.6996953888169183 0.7449093980546128


In [28]:
test = pd.read_csv("./TransTCR_test/McPAS/pair_major.csv").fillna("None")
test_data_iter = Data.DataLoader(TCR_DataSet(test), batch_size = len(test["TCRB"].to_list()), shuffle = False)
for step, (data, label) in enumerate(test_data_iter):
    tokens_TCR_res = data[0]
    segments_TCR_res = data[1]
    valid_len_TCR_res = data[2]
    tokens_MHC_res = data[3]
    segments_MHC_res = data[4]
    valid_len_MHC_res = data[5]
    tokens_peptide_res = data[6]
    segments_peptide_res = data[7]
    valid_len_peptide_res = data[8]
    valid_len_pMT_res = data[9]
    valid_len_pMHC_res = data[10]
    labels = label
    model.to(device)
    model.eval()
    with torch.no_grad():
        encoder_TCR, _, _ = TCR_encoder(tokens_TCR_res.to(device), 
                                        segments_TCR_res.to(device), 
                                        valid_len_TCR_res.to(device), 
                                        None)
        _, encoder_pMHC, decoder_pMHC = pMHC_encoder(tokens_MHC_res.to(device), 
                                                     segments_MHC_res.to(device), 
                                                     valid_len_MHC_res.to(device),
                                                     tokens_peptide_res.to(device), 
                                                     segments_peptide_res.to(device), 
                                                     valid_len_peptide_res.to(device),
                                                     valid_len_pMHC_res.to(device))
        output = model(encoder_TCR.to(device), encoder_pMHC.to(device), valid_len_pMT_res.to(device))
        output_softmax = output
        labels = labels.to(device)
        _, output = torch.max(output, 1)
conf_mat = confusion_matrix(labels, output)
confusion_matrix1 = confusion_matrix(labels, output)
new_confusion_matrix1 = [[confusion_matrix1[1,1], confusion_matrix1[1,0]],
                         [confusion_matrix1[0,1], confusion_matrix1[0,0]]]
accuracy = accuracy_score(labels, output) #准确率
precision = precision_score(labels, output) #精确率
recall = recall_score(labels, output) #召回率
f1= f1_score(labels, output) #F1
MCC = matthews_corrcoef(labels, output) #MCC
fpr, tpr, threshold = roc_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_auc = auc(fpr, tpr)
precision1, recall1, threshold = precision_recall_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_aupr = auc(recall1, precision1)
print("-----------------------------------------------------------------")
print("pair uncommon")
print(new_confusion_matrix1)
print("accuracy, precision, recall, f1, MCC, AUC, AUPR")
print(accuracy, precision, recall, f1, MCC, roc_auc, roc_aupr)

test = pd.read_csv("./TransTCR_test/McPAS/pair_major.csv")
test["Prediction"] = output
test["Score"] = output_softmax[:,1].to("cpu").detach().numpy()
pd.DataFrame(test).to_csv("./TransTCR_test/McPAS/pair_major.csv", index=False)

torch.Size([298, 6208])
-----------------------------------------------------------------
pair uncommon
[[104, 45], [30, 119]]
accuracy, precision, recall, f1, MCC, AUC, AUPR
0.7483221476510067 0.7761194029850746 0.697986577181208 0.7349823321554771 0.4991802526593649 0.7938381154002072 0.8067829931145684


In [29]:
test = pd.read_csv("./TransTCR_test/McPAS/pair_few.csv").fillna("None")
test_data_iter = Data.DataLoader(TCR_DataSet(test), batch_size = len(test["TCRB"].to_list()), shuffle = False)
for step, (data, label) in enumerate(test_data_iter):
    tokens_TCR_res = data[0]
    segments_TCR_res = data[1]
    valid_len_TCR_res = data[2]
    tokens_MHC_res = data[3]
    segments_MHC_res = data[4]
    valid_len_MHC_res = data[5]
    tokens_peptide_res = data[6]
    segments_peptide_res = data[7]
    valid_len_peptide_res = data[8]
    valid_len_pMT_res = data[9]
    valid_len_pMHC_res = data[10]
    labels = label
    model.to(device)
    model.eval()
    with torch.no_grad():
        encoder_TCR, _, _ = TCR_encoder(tokens_TCR_res.to(device), 
                                        segments_TCR_res.to(device), 
                                        valid_len_TCR_res.to(device), 
                                        None)
        _, encoder_pMHC, decoder_pMHC = pMHC_encoder(tokens_MHC_res.to(device), 
                                                     segments_MHC_res.to(device), 
                                                     valid_len_MHC_res.to(device),
                                                     tokens_peptide_res.to(device), 
                                                     segments_peptide_res.to(device), 
                                                     valid_len_peptide_res.to(device),
                                                     valid_len_pMHC_res.to(device))
        output = model(encoder_TCR.to(device), encoder_pMHC.to(device), valid_len_pMT_res.to(device))
        output_softmax = output
        labels = labels.to(device)
        _, output = torch.max(output, 1)
conf_mat = confusion_matrix(labels, output)
confusion_matrix1 = confusion_matrix(labels, output)
new_confusion_matrix1 = [[confusion_matrix1[1,1], confusion_matrix1[1,0]],
                         [confusion_matrix1[0,1], confusion_matrix1[0,0]]]
accuracy = accuracy_score(labels, output) #准确率
precision = precision_score(labels, output) #精确率
recall = recall_score(labels, output) #召回率
f1= f1_score(labels, output) #F1
MCC = matthews_corrcoef(labels, output) #MCC
fpr, tpr, threshold = roc_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_auc = auc(fpr, tpr)
precision1, recall1, threshold = precision_recall_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_aupr = auc(recall1, precision1)
print("-----------------------------------------------------------------")
print("pdb beta")
print(new_confusion_matrix1)
print("accuracy, precision, recall, f1, MCC, AUC, AUPR")
print(accuracy, precision, recall, f1, MCC, roc_auc, roc_aupr)

test = pd.read_csv("./TransTCR_test/McPAS/pair_few.csv")
test["Prediction"] = output
test["Score"] = output_softmax[:,1].to("cpu").detach().numpy()
pd.DataFrame(test).to_csv("./TransTCR_test/McPAS/pair_few.csv", index=False)

torch.Size([438, 6208])
-----------------------------------------------------------------
pdb beta
[[183, 36], [85, 134]]
accuracy, precision, recall, f1, MCC, AUC, AUPR
0.723744292237443 0.6828358208955224 0.8356164383561644 0.7515400410677618 0.4591284639882931 0.78215633535581 0.7731564617382265


In [30]:
test = pd.read_csv("./TransTCR_test/McPAS/pair_zero.csv").fillna("None")
test_data_iter = Data.DataLoader(TCR_DataSet(test), batch_size = len(test["TCRB"].to_list()), shuffle = False)
for step, (data, label) in enumerate(test_data_iter):
    tokens_TCR_res = data[0]
    segments_TCR_res = data[1]
    valid_len_TCR_res = data[2]
    tokens_MHC_res = data[3]
    segments_MHC_res = data[4]
    valid_len_MHC_res = data[5]
    tokens_peptide_res = data[6]
    segments_peptide_res = data[7]
    valid_len_peptide_res = data[8]
    valid_len_pMT_res = data[9]
    valid_len_pMHC_res = data[10]
    labels = label
    model.to(device)
    model.eval()
    with torch.no_grad():
        encoder_TCR, _, _ = TCR_encoder(tokens_TCR_res.to(device), 
                                        segments_TCR_res.to(device), 
                                        valid_len_TCR_res.to(device), 
                                        None)
        _, encoder_pMHC, decoder_pMHC = pMHC_encoder(tokens_MHC_res.to(device), 
                                                     segments_MHC_res.to(device), 
                                                     valid_len_MHC_res.to(device),
                                                     tokens_peptide_res.to(device), 
                                                     segments_peptide_res.to(device), 
                                                     valid_len_peptide_res.to(device),
                                                     valid_len_pMHC_res.to(device))
        output = model(encoder_TCR.to(device), encoder_pMHC.to(device), valid_len_pMT_res.to(device))
        output_softmax = output
        labels = labels.to(device)
        _, output = torch.max(output, 1)
conf_mat = confusion_matrix(labels, output)
confusion_matrix1 = confusion_matrix(labels, output)
new_confusion_matrix1 = [[confusion_matrix1[1,1], confusion_matrix1[1,0]],
                         [confusion_matrix1[0,1], confusion_matrix1[0,0]]]
accuracy = accuracy_score(labels, output) #准确率
precision = precision_score(labels, output) #精确率
recall = recall_score(labels, output) #召回率
f1= f1_score(labels, output) #F1
MCC = matthews_corrcoef(labels, output) #MCC
fpr, tpr, threshold = roc_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_auc = auc(fpr, tpr)
precision1, recall1, threshold = precision_recall_curve(labels, output_softmax[:,1].to("cpu").detach().numpy())
roc_aupr = auc(recall1, precision1)
print("-----------------------------------------------------------------")
print("pdb pair")
print(new_confusion_matrix1)
print("accuracy, precision, recall, f1, MCC, AUC, AUPR")
print(accuracy, precision, recall, f1, MCC, roc_auc, roc_aupr)

test = pd.read_csv("./TransTCR_test/McPAS/pair_zero.csv")
test["Prediction"] = output
test["Score"] = output_softmax[:,1].to("cpu").detach().numpy()
pd.DataFrame(test).to_csv("./TransTCR_test/McPAS/pair_zero.csv", index=False)

torch.Size([1186, 6208])
-----------------------------------------------------------------
pdb pair
[[395, 198], [131, 462]]
accuracy, precision, recall, f1, MCC, AUC, AUPR
0.7225969645868465 0.7509505703422054 0.6661045531197302 0.7059874888293118 0.4480630048037489 0.7464531393520243 0.7658203831926876
